# Step 1- Imports

In [42]:
import numpy as np
import pandas as pd
from pathlib import Path

print("Imports OK ")

Imports OK 


# Step 2 - Configuration

In [43]:
#  Paths 
DATA_PATH   = Path("../data/features/feature_matrix.csv")
OUTPUT_DIR  = Path("../data/split")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#  Constants 
N_INTERVALS   = 96          # quarter-hours per day (24h x 4)
INITIAL_TRAIN = 100         # days in first training window
EVAL_DAYS     = 44          # out-of-sample evaluation days


FEATURES = [
    "price_lag1d",
    "price_lag7d",
    "price_hourly_lag1d",
    "price_hourly_lag7d",
    "wind_mwh",
    "solar_mwh",
    "load_mwh",
]
TARGET = "price_eur_mwh"

print(f"Data path    : {DATA_PATH}")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"Train window : {INITIAL_TRAIN} days")
print(f"Eval window  : {EVAL_DAYS} days")

Data path    : ..\data\features\feature_matrix.csv
Output dir    : ..\data\split
Train window : 100 days
Eval window  : 44 days


# Step 3 - Load Feature Matrix 

In [44]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Shape        : {df.shape}")
print(f"Date range   : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print(f"Unique days  : {df['timestamp'].dt.date.nunique()}")
print()
print(df[FEATURES + [TARGET]].describe().round(2))

Shape        : (13836, 9)
Date range   : 2025-10-08 → 2026-02-28
Unique days  : 144

       price_lag1d  price_lag7d  price_hourly_lag1d  price_hourly_lag7d  \
count     13836.00     13836.00            13836.00            13836.00   
mean         98.97        98.36              111.48              109.68   
std          40.88        42.50               64.84               65.17   
min          -1.05        -5.09              -15.69              -15.69   
25%          82.35        82.18               83.01               80.88   
50%          95.58        95.44              108.29              106.21   
75%         113.03       113.15              132.19              130.09   
max         508.38       508.38              936.28              936.28   

       wind_mwh  solar_mwh  load_mwh  price_eur_mwh  
count  13836.00   13836.00  13836.00       13836.00  
mean    5247.15     791.56  14462.79          98.38  
std     2775.51    1470.29   2199.25          40.73  
min      111.14       0

# Step 4 — DST Correction

The interval verification identified one day with an unexpected interval count:
**26 October 2025 contains 108 quarter-hourly intervals** instead of the expected 96.
This is caused by the daylight saving time (DST) transition in Germany, where clocks are set back one hour on that date (from CEST to CET), making the calendar day 25 hours long. The SMARD platform recorded all 108 quarter-hourly intervals for that day, including the repeated hour.

This extra hour introduces an inconsistency into the feature matrix: all other days have exactly 96 rows, but this day has 12 additional rows. If left uncorrected, this would cause misalignment in the rolling window evaluation, since the models assume a fixed structure of 96 observations per day.

To correct this, only the first 96 intervals of 26 October 2025 are retained and the remaining 12 rows are discarded. This reduces the total row count from 13,836 to 13,824, consistent with 144 complete days of 96 intervals each.

In [45]:
# Check intervals per day before fix 
counts   = df.groupby(df["timestamp"].dt.date).size()
bad_days = counts[counts != N_INTERVALS]

print("Before fix:")
if len(bad_days) == 0:
    print(f"   All {len(counts)} days have exactly {N_INTERVALS} intervals")
else:
    print(f"  Days with unexpected interval count:")
    print(bad_days.to_string())

Before fix:
  Days with unexpected interval count:
timestamp
2025-10-26    108


In [46]:
#  Apply DST fix: keep only first 96 intervals of 26 Oct 2025
dst_day  = pd.Timestamp("2025-10-26").date()
dst_mask = df["timestamp"].dt.date == dst_day

df_dst  = df[dst_mask].sort_values("timestamp").iloc[:N_INTERVALS]
df_rest = df[~dst_mask]

df = pd.concat([df_rest, df_dst]).sort_values("timestamp").reset_index(drop=True)

#  Verify fix
counts   = df.groupby(df["timestamp"].dt.date).size()
bad_days = counts[counts != N_INTERVALS]

print("After fix:")
print(f"  Rows        : {len(df):,}")
print(f"  Unique days : {df['timestamp'].dt.date.nunique()}")
if len(bad_days) == 0:
    print(f"   All {len(counts)} days have exactly {N_INTERVALS} intervals")
else:
    print(f"   Remaining problem days:")
    print(bad_days.to_string())

After fix:
  Rows        : 13,824
  Unique days : 144
   All 144 days have exactly 96 intervals


# Step 5 — Train / Test Split

Following the expanding-window scheme described in Section 3.4:
- **Initial training window:** first 100 days (8 Oct 2025 – 15 Jan 2026)
- **Out-of-sample evaluation:** remaining 44 days (16 Jan 2026 – 28 Feb 2026)

All four models in Chapter 4 will be evaluated over the same 44-day period and receive identical input features at every forecast origin.

In [47]:
days       = sorted(df["timestamp"].dt.date.unique())
n_days     = len(days)

train_days = days[:INITIAL_TRAIN]
test_days  = days[INITIAL_TRAIN:]

print("=" * 55)
print("  TRAIN – TEST SPLIT SUMMARY")
print("=" * 55)
print(f"  Total days         : {n_days}")
print(f"  Training window    : {len(train_days)} days")
print(f"    From             : {train_days[0]}")
print(f"    To               : {train_days[-1]}")
print(f"    Observations     : {len(train_days) * N_INTERVALS:,}")
print()
print(f"  Evaluation window  : {len(test_days)} days")
print(f"    From             : {test_days[0]}")
print(f"    To               : {test_days[-1]}")
print(f"    Observations     : {len(test_days) * N_INTERVALS:,}")
print()
print(f"  Total observations : {n_days * N_INTERVALS:,}")
print("=" * 55)

  TRAIN – TEST SPLIT SUMMARY
  Total days         : 144
  Training window    : 100 days
    From             : 2025-10-08
    To               : 2026-01-15
    Observations     : 9,600

  Evaluation window  : 44 days
    From             : 2026-01-16
    To               : 2026-02-28
    Observations     : 4,224

  Total observations : 13,824


# Step 6 — Save the Split

Save the corrected feature matrix and the train/test day lists so that `models.ipynb` can load them directly without repeating any of these steps.

In [48]:
# Save corrected feature matrix
df.to_csv(OUTPUT_DIR / "feature_matrix_clean.csv", index=False)
print(f"Saved: feature_matrix_clean.csv  ({len(df):,} rows)")

# Save train day list 
pd.Series([str(d) for d in train_days]).to_csv(
    OUTPUT_DIR / "train_days.csv", index=False, header=["date"]
)
print(f"Saved: train_days.csv            ({len(train_days)} days)")

# Save test day list 
pd.Series([str(d) for d in test_days]).to_csv(
    OUTPUT_DIR / "test_days.csv", index=False, header=["date"]
)
print(f"Saved: test_days.csv             ({len(test_days)} days)")

print()
print(" models.ipynb can now load these files directly.")

Saved: feature_matrix_clean.csv  (13,824 rows)
Saved: train_days.csv            (100 days)
Saved: test_days.csv             (44 days)

 models.ipynb can now load these files directly.


# Step 7 — Final Verification

In [49]:
print("=" * 55)
print("FINAL VERIFICATION")
print("=" * 55)
print()

# No data leakage check
last_train = train_days[-1]
first_test = test_days[0]
gap_days   = (pd.Timestamp(first_test) - pd.Timestamp(last_train)).days
print(f"  Last training day  : {last_train}")
print(f"  First test day     : {first_test}")
print(f"  Gap between them   : {gap_days} day(s)")
if gap_days == 1:
    print("No overlap between train and test — no data leakage")
else:
    print("Unexpected gap — check split")

print()

# Feature completeness check
missing = df[FEATURES + [TARGET]].isna().sum()
if missing.sum() == 0:
    print("No missing values in features or target")
else:
    print("Missing values found:")
    print(missing[missing > 0])

print()
print("Section 3.4 complete — ready for models.ipynb")
print("=" * 55)

FINAL VERIFICATION

  Last training day  : 2026-01-15
  First test day     : 2026-01-16
  Gap between them   : 1 day(s)
No overlap between train and test — no data leakage

No missing values in features or target

Section 3.4 complete — ready for models.ipynb
